# 03 — Blended LLM + ML (Groq · tune / validate)

Blends **XGBoost** (calibrated structured-feature model) with a **Llama-3.3-70B LLM via Groq**
to maximise **Charged-Off F1** — the metric that matters, since Charged Off is the minority class.

### What changed vs. the old version
- **LLM is Groq** (`llama-3.3-70b-versatile`), called directly from this notebook — no dependency
  on pre-computed prediction CSVs.
- **Three samples, proper split** — no more optimising and reporting on the same 100 loans:

| Sample | Role | Used for |
|--------|------|----------|
| `tuning_sample` | **train / tune** | grid-search every blend hyper-parameter (α, threshold, gate band) |
| `robustness_batch` | **validation** | pick the winning *strategy* (locked params) |
| `test_batch` | **test — strictly held out** | evaluated in Phase 04 final benchmark |

Hyper-parameters are fit on **tuning** only, the best strategy is chosen on **robustness**, and
**robustness** is used to select the winning strategy. The untouched test set is evaluated in Phase 04 final benchmark.
gains are held-out, not in-sample.

### LLM signals generated per loan
1. **Binary** — `top_features_only` prompt (the Phase-1 winner): 8 key features → {Fully Paid, Charged Off}.
2. **5A risk score** — borrower *description text only* → 0–10 risk (Best-of-N self-consistency).
3. **5B risk score** — description + structured features → 0–10 risk (Best-of-N).

### Strategies on the leaderboard
Hard-label rules (union/intersection/boost/prune) · soft blend (binary) · confidence gate ·
risk-score blends (5A / 5B) · optional sentence-embedding blend.


In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import re
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, confusion_matrix,
)

from llm_utils import (
    run_ml_on_sample, evaluate_predictions, call_llm, load_api_key,
    parse_llm_response, build_system_prompt, build_user_prompt,
    format_loan_features, FEATURE_DESCRIPTIONS,
    RESULTS_DIR, DATA_DIR, MODEL_DIR,
)

# ── LLM provider ──────────────────────────────────────────────────────────────
LLM_PROVIDER = 'groq'
LLM_MODEL    = 'llama-3.3-70b-versatile'

# Best-of-N self-consistency for the numeric risk scorer (Wang et al., 2022):
# sample N times at low temperature, average the scores → lower single-call variance.
BON_N    = 3
BON_TEMP = 0.3

# Set True to ignore cached Groq signal CSVs and re-call the API.
FORCE_REGEN = False


def metrics_dict(name, y_true, y_pred):
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    return {
        'model':        name,
        'accuracy':     round(accuracy_score(yt, yp), 3),
        'precision_co': round(precision_score(yt, yp, pos_label=0, zero_division=0), 3),
        'recall_co':    round(recall_score(yt, yp, pos_label=0, zero_division=0), 3),
        'f1_co':        round(f1_score(yt, yp, pos_label=0, zero_division=0), 3),
    }


print('Imports OK.')
print(f'LLM: {LLM_PROVIDER} | {LLM_MODEL}')
_key = load_api_key(LLM_PROVIDER)
print('Groq API key:', 'loaded' if _key else 'MISSING - add GROQ_API_KEY to notebooks/llm_models/.env')
print(f'DATA_DIR:    {DATA_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')


### Load XGBoost

Loads the saved `xgb_model.joblib` + `thresholds.joblib` (tracked in git). If missing, run
`notebooks/ml_models/03_Modeling.ipynb` first. `run_ml_on_sample` re-encodes/scales each sample


In [ ]:
xgb_model_path = os.path.join(MODEL_DIR, 'xgb_model.joblib')
thresh_path    = os.path.join(MODEL_DIR, 'thresholds.joblib')

assert os.path.exists(xgb_model_path) and os.path.exists(thresh_path), (
    'xgb_model.joblib / thresholds.joblib not found in models/. '
    'Run notebooks/ml_models/03_Modeling.ipynb first.'
)

xgb_threshold = joblib.load(thresh_path)['xgb']
print(f'XGBoost ready | decision threshold = {xgb_threshold:.3f}')
print('Convention: run_ml_on_sample returns P(Fully Paid). pred = 1 (FP) if P >= threshold, else 0 (CO).')


## Datasets

The three role-based samples from `data/processed/` (all 100 loans, same 35-column schema,


In [ ]:
SAMPLE_FILES = {
    'tuning':     'tuning_sample.csv',      # train / tune hyper-parameters
    'robustness': 'robustness_batch.csv',   # validation — select strategy
}

SAMPLES = {}
for name, fname in SAMPLE_FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    y_true = df['loan_status'].values.astype(int)
    xgb_probs, xgb_preds = run_ml_on_sample(df)
    SAMPLES[name] = {
        'df': df, 'y_true': y_true,
        'xgb_probs': np.asarray(xgb_probs), 'xgb_preds': np.asarray(xgb_preds),
    }
    co = int((y_true == 0).sum())
    print(f'{name:11s}: {len(df):3d} loans | CO={co:2d} FP={len(df)-co:2d} '
          f'| XGB flags {int((xgb_preds==0).sum())} CO  '
          f'(solo CO-F1 {f1_score(y_true, xgb_preds, pos_label=0, zero_division=0):.3f})')

TUNING = SAMPLES['tuning']  # convenience handle for the cells that tune
print('\\nAll hyper-parameters below are tuned on `tuning` only.')


## Groq LLM signals  (batched — Groq free tier is 30 req/min, 1000 req/day)

For every loan we generate three signals from Groq and **cache them to CSV**
(`06_groq_signals_<sample>.csv`). Re-running reloads the cache instead of re-calling the API;
set `FORCE_REGEN = True` (Setup cell) to force fresh calls.

**Requests are batched** — many loans per call, not one call per loan — using the repo's
TOON-style compact encoding. That keeps us well under the rate limits:

| Signal | Naive (1/loan) | Batched |
|--------|----------------|---------|
| Binary (`top_features_only`) | 100 | ~2 |
| 5A risk score ×Best-of-N | 300 | ~6 |
| 5B risk score ×Best-of-N | 300 | ~15 |
| **per 100-loan sample** | **700** | **~23** |

So all three samples cost ~70 requests total (vs the 1,000/day cap). The binary signal reuses the


In [ ]:
# ── Batched signal generation (Groq free tier: 30 RPM / 1000 RPD / 12000 TPM) ──
# One request per CHUNK of loans, not per loan. Collapses ~700 calls/sample to ~9-23.
import math
from llm_utils import LLM_FEATURES

BATCH_BINARY = 50    # loans per binary request
BATCH_5A     = 50    # loans per 5A request   (description text only)
BATCH_5B     = 20    # loans per 5B request   (full features -> heavier, smaller chunks)
THROTTLE     = 2.0   # seconds between requests, to stay under 30 RPM

TOP_FEATURES = [
    'int_rate', 'sub_grade', 'dti', 'revol_util',
    'annual_inc', 'installment', 'loan_amnt', 'revol_bal',
]

def _has_desc(row):
    d = row.get('desc', '')
    return bool(d and not (isinstance(d, float) and np.isnan(d)) and str(d).strip())

def _feat_val(row, f):
    v = row.get(f)
    if pd.isna(v):
        return 'NA'
    if f == 'mths_since_last_delinq' and v == 999:   # preprocessing sentinel
        return 'none'
    return str(v)

def _format_top_features(row):
    # Same labelled 8-feature view the Phase-1 winner used.
    out = []
    for feat in TOP_FEATURES:
        if feat in row and pd.notna(row[feat]):
            out.append(f"- {FEATURE_DESCRIPTIONS.get(feat, feat)}: {row[feat]}")
    return "\n".join(out)

def _parse_int_array(text, n, key):
    text = (text or '').strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
        text = text.strip()
    out = [None] * n
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for k in ('predictions', 'results', 'loans', 'data', 'scores'):
                if k in data and isinstance(data[k], list):
                    data = data[k]
                    break
        for item in data:
            idx = item.get('i', item.get('index'))
            if idx is None:
                continue
            idx = int(idx)
            v = item.get(key)
            if 0 <= idx < n and v is not None:
                out[idx] = v
    except Exception:
        pass
    return out

# ── 1. Binary prediction (top_features_only) — batched, NO per-loan reasoning ──
# Reasoning is dropped on purpose: the blend only uses the 0/1, and asking for
# prose per loan makes the batch output long -> truncation/parse failures. The
# *input* still matches the winning prompt (labelled 8 features); only the output
# is lean (one {"i","prediction"} per loan), which parses reliably.
def run_binary_batched(df):
    n = len(df)
    preds = [None] * n
    sys_p = build_system_prompt()
    for s in range(0, n, BATCH_BINARY):
        sub = df.iloc[s:s + BATCH_BINARY]
        lines = [
            f"Predict the outcome (1 = Fully Paid, 0 = Charged Off) for EACH of the {len(sub)} loans.",
            'Return ONLY a JSON array, one object per loan, every index included:',
            '[{"i": 0, "prediction": 1}, {"i": 1, "prediction": 0}, ...]', "",
        ]
        for i, (_, row) in enumerate(sub.iterrows()):
            lines.append(f"--- LOAN #{i} (key features) ---")
            lines.append(_format_top_features(row))
        raw = call_llm(sys_p, "\n".join(lines), api_provider=LLM_PROVIDER,
                       model=LLM_MODEL, temperature=0.0, max_tokens=len(sub) * 15 + 200)
        for j, v in enumerate(_parse_int_array(raw, len(sub), 'prediction')):
            preds[s + j] = (int(v) if v in (0, 1, '0', '1') else None)
        time.sleep(THROTTLE)
    return preds

# ── 2 & 3. Numeric risk scorer (0-10), batched + Best-of-N self-consistency ───
_SCORE_GUIDE = (
    '0-3 = low risk (specific purpose, professional tone, clear repayment plan); '
    '4-6 = moderate (vague / generic / neutral); '
    '7-10 = high risk (desperate tone, inconsistent story, alarming language).'
)

def _risk_batch_prompt(sub, mode):
    n = len(sub)
    head = [
        f"Rate the default risk 0-10 for EACH of the {n} loans below.",
        'Return ONLY a JSON array, one object per loan, every index included:',
        '[{"i": 0, "risk_score": <0-10 number>}, ...]',
        _SCORE_GUIDE, "",
    ]
    if mode == 'desc':
        for i, (_, row) in enumerate(sub.iterrows()):
            d = str(row.get('desc', '')).strip() if _has_desc(row) else '(no description)'
            head.append(f'#{i}: "{d}"')
    else:
        # 5B: TOON-style compact table. Field names ONCE, then one pipe-delimited
        # row per loan -> ~3-4x fewer input tokens than repeating 30 labels/loan.
        # Pure prompt text, so it also optimizes the paid OpenAI run later.
        feats = [f for f in LLM_FEATURES if f in sub.columns]
        head.append("Each row is one loan, pipe-separated, in this exact column order:")
        head.append("index | " + " | ".join(feats) + " | borrower_desc")
        for i, (_, row) in enumerate(sub.iterrows()):
            vals = " | ".join(_feat_val(row, f) for f in feats)
            d = str(row.get('desc', '')).strip() if _has_desc(row) else '(none)'
            head.append(f'#{i} | {vals} | "{d}"')
    return "\n".join(head)

def score_risk_batched(df, mode, chunk):
    n = len(df)
    sys_p = ('You are a credit risk analyst.' if mode == 'desc'
             else 'You are a credit risk analyst reviewing full loan applications.')
    runs = []
    for _ in range(BON_N):                       # Best-of-N: N independent passes
        run = [None] * n
        for s in range(0, n, chunk):
            sub = df.iloc[s:s + chunk]
            raw = call_llm(sys_p, _risk_batch_prompt(sub, mode),
                           api_provider=LLM_PROVIDER, model=LLM_MODEL,
                           temperature=BON_TEMP, max_tokens=len(sub) * 25 + 200)
            for j, v in enumerate(_parse_int_array(raw, len(sub), 'risk_score')):
                run[s + j] = (min(10.0, max(0.0, float(v))) if v is not None else None)
            time.sleep(THROTTLE)
        runs.append(run)
    out = []                                      # per-loan average across passes
    for i in range(n):
        vals = [r[i] for r in runs if r[i] is not None]
        out.append(float(np.mean(vals)) if vals else 5.0)  # 5.0 = neutral fallback
    return np.array(out)

_est = (math.ceil(100 / BATCH_BINARY)
        + BON_N * math.ceil(100 / BATCH_5A)
        + BON_N * math.ceil(100 / BATCH_5B))
print(f'Batched signal functions ready. ~{_est} Groq requests per 100-loan sample (was 700).')
print(f'Provider={LLM_PROVIDER}, model={LLM_MODEL}, Best-of-N={BON_N} @ temp={BON_TEMP}, throttle={THROTTLE}s.')


In [ ]:
# Per-signal caching: each of {binary, 5a, 5b} is cached separately as
# 06_groq_<sig>_<sample>.csv. If a later signal hits the daily token wall, the
# earlier ones are already on disk, so a re-run resumes (no wasted tokens).
def _sig_cache(name, sig):
    return os.path.join(RESULTS_DIR, f'03_groq_{sig}_{name}.csv')

def generate_signals(name, df, force=False):
    n = len(df)
    y = df['loan_status'].astype(int).values
    specs = [
        ('binary', lambda: run_binary_batched(df)),
        ('5a',     lambda: score_risk_batched(df, 'desc',     BATCH_5A)),
        ('5b',     lambda: score_risk_batched(df, 'descfeat', BATCH_5B)),
    ]
    vals = {}
    for sig, fn in specs:
        cp = _sig_cache(name, sig)
        if os.path.exists(cp) and not force:
            col = pd.read_csv(cp)
            if len(col) == n:
                vals[sig] = col['value'].values
                print(f'[{name}/{sig}] cached ({n} rows)')
                continue
        print(f'[{name}/{sig}] generating via Groq...')
        v = fn()
        os.makedirs(RESULTS_DIR, exist_ok=True)
        pd.DataFrame({'loan_index': range(n), 'value': v}).to_csv(cp, index=False)
        vals[sig] = np.asarray(v, dtype=object)
        print(f'[{name}/{sig}] saved 06_groq_{sig}_{name}.csv')

    out = pd.DataFrame({
        'loan_index': range(n), 'actual': y,
        'llm_binary': vals['binary'], 'score_5a': vals['5a'], 'score_5b': vals['5b'],
        'has_desc': [_has_desc(r) for _, r in df.iterrows()],
    })
    out.to_csv(os.path.join(RESULTS_DIR, f'03_groq_signals_{name}.csv'), index=False)
    return out


for name in SAMPLES:
    sig = generate_signals(name, SAMPLES[name]['df'], force=FORCE_REGEN)

    assert (sig['actual'].values == SAMPLES[name]['y_true']).all(), (
        f'{name}: cached signal labels do not match the sample - set FORCE_REGEN=True'
    )

    raw_bin = [None if (isinstance(x, float) and np.isnan(x)) else int(x)
               for x in pd.to_numeric(sig['llm_binary'], errors='coerce').tolist()]
    xgb_preds = SAMPLES[name]['xgb_preds']
    filled = np.array([xgb_preds[j] if raw_bin[j] is None else raw_bin[j]
                       for j in range(len(raw_bin))])

    SAMPLES[name]['llm_binary_raw'] = raw_bin
    SAMPLES[name]['llm_binary']     = filled
    SAMPLES[name]['score_5a']       = sig['score_5a'].values.astype(float)
    SAMPLES[name]['score_5b']       = sig['score_5b'].values.astype(float)
    SAMPLES[name]['has_desc']       = sig['has_desc'].values.astype(bool)
    n_missing = sum(1 for x in raw_bin if x is None)


---
## Part 1 — Baselines


In [ ]:
baseline_rows = []
for name in SAMPLES:
    S = SAMPLES[name]
    baseline_rows.append({'sample': name, **metrics_dict('XGBoost', S['y_true'], S['xgb_preds'])})
    baseline_rows.append({'sample': name, **metrics_dict('LLM (Groq, top_features)', S['y_true'], S['llm_binary'])})

baseline_df = pd.DataFrame(baseline_rows).set_index(['sample', 'model'])
print(baseline_df.to_string())

# Headline solo numbers from the TUNING sample (used as reference lines in plots).
xgb_m = metrics_dict('XGBoost', TUNING['y_true'], TUNING['xgb_preds'])
llm_m = metrics_dict('LLM',     TUNING['y_true'], TUNING['llm_binary'])
print(f"\n[tuning] XGB solo CO-F1={xgb_m['f1_co']:.3f}  |  LLM solo CO-F1={llm_m['f1_co']:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
names = list(SAMPLES)
x = np.arange(len(names)); w = 0.35
xgb_f1 = [f1_score(SAMPLES[n]['y_true'], SAMPLES[n]['xgb_preds'], pos_label=0, zero_division=0) for n in names]
llm_f1 = [f1_score(SAMPLES[n]['y_true'], SAMPLES[n]['llm_binary'], pos_label=0, zero_division=0) for n in names]
b1 = ax.bar(x - w/2, xgb_f1, w, label='XGBoost', color='#2196f3')
b2 = ax.bar(x + w/2, llm_f1, w, label='LLM (Groq)', color='#f4a529')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('Charged-Off F1'); ax.set_ylim(0, max(xgb_f1+llm_f1)+0.1)
ax.set_title('Part 1 - Solo baselines per sample'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '03_baselines.png'), dpi=150, bbox_inches='tight'); plt.show()


---
## Part 2 — Hard-label ensemble rules
Boolean combinations of the two binary predictions. Union/Intersection need no tuning;


In [ ]:
BOOST_BAND = 0.15   # XGB borderline window for the LLM-boost rule
PRUNE_OFF  = 0.25   # XGB high-confidence offset for the LLM-prune rule

def hard_label_preds(S):
    xp, xpr, lb = S['xgb_preds'], S['xgb_probs'], S['llm_binary']
    out = {}
    out['Union (XGB OR LLM)']        = np.where((xp == 0) | (lb == 0), 0, 1)
    out['Intersection (XGB AND LLM)'] = np.where((xp == 0) & (lb == 0), 0, 1)
    borderline = (xpr > xgb_threshold - BOOST_BAND) & (xpr < xgb_threshold + BOOST_BAND)
    boost = xp.copy(); boost[(xp == 1) & (lb == 0) & borderline] = 0
    out['XGB + LLM borderline boost'] = boost
    prune = lb.copy(); prune[(lb == 0) & (xpr >= xgb_threshold + PRUNE_OFF)] = 1
    out['LLM + XGB high-conf prune'] = prune
    return out

hard_rows = []
for name in SAMPLES:
    S = SAMPLES[name]
    hard_rows.append({'sample': name, **metrics_dict('XGBoost (solo)', S['y_true'], S['xgb_preds'])})
    hard_rows.append({'sample': name, **metrics_dict('LLM (solo)',     S['y_true'], S['llm_binary'])})
    for rule, preds in hard_label_preds(S).items():
        hard_rows.append({'sample': name, **metrics_dict(rule, S['y_true'], preds)})

hard_df = pd.DataFrame(hard_rows).set_index(['sample', 'model'])
print(hard_df.to_string())


---
## Part 3 — Soft blend (binary LLM)
`score = α·P_xgb(FP) + (1−α)·llm_binary`, predict FP if `score ≥ threshold`.

**α and threshold are grid-searched on `tuning` only**, then locked and applied unchanged to


In [ ]:
alphas      = np.round(np.arange(0.00, 1.01, 0.05), 2)
thresh_vals = np.round(np.unique(np.append(np.arange(0.05, 0.96, 0.05), xgb_threshold)), 3)

def soft_preds(S, a, t, llm_key='llm_binary'):
    return ((a * S['xgb_probs'] + (1.0 - a) * S[llm_key].astype(float)) >= t).astype(int)

def tune_soft(S, llm_key='llm_binary'):
    best = (-1, 1.0, 0.5)  # (f1, alpha, threshold)
    for a in alphas:
        for t in thresh_vals:
            f1 = f1_score(S['y_true'], soft_preds(S, a, t, llm_key), pos_label=0, zero_division=0)
            if f1 > best[0]:
                best = (f1, a, t)
    return best

_f1, BLEND_A, BLEND_T = tune_soft(TUNING)
print(f'Tuned on `tuning`: alpha={BLEND_A:.2f}  threshold={BLEND_T:.2f}  (tuning CO-F1={_f1:.3f})')
print(f'  alpha=1 -> XGB only, alpha=0 -> LLM only.  Locking these and applying to all samples:\n')

for name in SAMPLES:
    S = SAMPLES[name]
    p = soft_preds(S, BLEND_A, BLEND_T)
    S['soft_blend_preds'] = p
    m = metrics_dict('soft blend', S['y_true'], p)
    print(f'  {name:11s}  CO-F1={m["f1_co"]:.3f}  recall={m["recall_co"]:.3f}  prec={m["precision_co"]:.3f}  acc={m["accuracy"]:.3f}')


In [ ]:
# Heatmap of the tuning-sample grid (where the params were chosen).
grid = np.zeros((len(alphas), len(thresh_vals)))
for ia, a in enumerate(alphas):
    for it, t in enumerate(thresh_vals):
        grid[ia, it] = f1_score(TUNING['y_true'], soft_preds(TUNING, a, t), pos_label=0, zero_division=0)

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(grid, aspect='auto', origin='lower', cmap='RdYlGn',
               vmin=max(0.0, grid.min()), vmax=grid.max())
plt.colorbar(im, ax=ax, label='Charged-Off F1 (tuning)', shrink=0.85)
sc = max(1, len(thresh_vals)//10); sr = max(1, len(alphas)//10)
ax.set_xticks(range(0, len(thresh_vals), sc)); ax.set_xticklabels([f'{t:.2f}' for t in thresh_vals[::sc]], rotation=45, fontsize=7)
ax.set_yticks(range(0, len(alphas), sr));      ax.set_yticklabels([f'{a:.2f}' for a in alphas[::sr]], fontsize=7)
ax.set_xlabel('threshold'); ax.set_ylabel('alpha (0=LLM only, 1=XGB only)')
ax.set_title('Part 3 - Soft-blend grid on TUNING (star = locked optimum)')
ax.scatter([list(thresh_vals).index(BLEND_T)], [list(alphas).index(BLEND_A)],
           color='white', edgecolor='black', s=220, marker='*', zorder=5)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '03_soft_blend_heatmap.png'), dpi=150, bbox_inches='tight'); plt.show()


---
## Part 4 — Confidence-gated consultation
XGBoost decides alone when its probability is far from the boundary; the LLM is consulted only
for borderline loans. **The band width is tuned on `tuning`**, then locked.

```
P_xgb < threshold - band  -> CO     (XGB confident)
P_xgb >= threshold + band -> FP     (XGB confident)
otherwise                 -> LLM binary prediction
```


In [ ]:
band_widths = np.round(np.arange(0.00, 0.511, 0.01), 3)

def gate_preds(S, band):
    lo = max(0.0, xgb_threshold - band); hi = min(1.0, xgb_threshold + band)
    xpr, lb = S['xgb_probs'], S['llm_binary']
    return np.array([0 if xpr[i] < lo else (1 if xpr[i] >= hi else int(lb[i])) for i in range(len(xpr))]), lo, hi

def gate_pct_llm(S, band):
    lo = max(0.0, xgb_threshold - band); hi = min(1.0, xgb_threshold + band)
    xpr = S['xgb_probs']
    return float(np.mean((xpr >= lo) & (xpr < hi)) * 100)

# Tune band on tuning.
tune_curve = []
best_band, best_band_f1 = 0.0, -1.0
for band in band_widths:
    p, _, _ = gate_preds(TUNING, band)
    f1 = f1_score(TUNING['y_true'], p, pos_label=0, zero_division=0)
    tune_curve.append((band, f1, gate_pct_llm(TUNING, band)))
    if f1 > best_band_f1:
        best_band_f1, best_band = f1, band

GATE_BAND = best_band
print(f'Tuned on `tuning`: band=+/-{GATE_BAND:.3f}  (tuning CO-F1={best_band_f1:.3f}, '
      f'{gate_pct_llm(TUNING, GATE_BAND):.0f}% sent to LLM)\n')

for name in SAMPLES:
    S = SAMPLES[name]
    p, lo, hi = gate_preds(S, GATE_BAND)
    S['gate_preds'] = p
    m = metrics_dict('gate', S['y_true'], p)
    print(f'  {name:11s}  CO-F1={m["f1_co"]:.3f}  recall={m["recall_co"]:.3f}  '
          f'prec={m["precision_co"]:.3f}  LLM-consulted={gate_pct_llm(S, GATE_BAND):.0f}%')


In [ ]:
tc = np.array(tune_curve)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Part 4 - Confidence gate tuned on TUNING (threshold={xgb_threshold:.3f})', fontsize=12)
ax1.plot(tc[:, 0], tc[:, 1], 'b-o', markersize=3, label='CO-F1 (tuning)')
ax1.axhline(xgb_m['f1_co'], color='steelblue', ls='--', alpha=0.6, label=f"XGB solo {xgb_m['f1_co']:.3f}")
ax1.axhline(llm_m['f1_co'], color='#f4a529', ls='--', alpha=0.6, label=f"LLM solo {llm_m['f1_co']:.3f}")
ax1.axvline(GATE_BAND, color='red', ls=':', label=f'locked band {GATE_BAND:.3f}')
ax1.set_xlabel('band width (+/-)'); ax1.set_ylabel('CO-F1'); ax1.set_title('F1 vs band'); ax1.legend(fontsize=8)
ax2.plot(tc[:, 0], tc[:, 2], 'r-D', markersize=3)
ax2.axvline(GATE_BAND, color='red', ls=':')
ax2.set_xlabel('band width (+/-)'); ax2.set_ylabel('% loans sent to LLM'); ax2.set_title('LLM consultation rate (cost proxy)')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '03_confidence_gate.png'), dpi=150, bbox_inches='tight'); plt.show()


---
## Part 5 — Risk-score blends (5A / 5B)
The numeric Groq risk score (0–10) becomes a probability analog and blends *continuously* with
XGBoost — a genuinely soft blend (vs. Part 3's binary term).

```
p_fp = 1 − risk_score/10          # high risk -> low P(Fully Paid)
score = α·P_xgb(FP) + (1−α)·p_fp
```


In [ ]:
def add_pfp(S):
    S['p_fp_5a'] = 1.0 - S['score_5a'] / 10.0
    S['p_fp_5b'] = 1.0 - S['score_5b'] / 10.0
for name in SAMPLES:
    add_pfp(SAMPLES[name])

def soft_preds_pfp(S, a, t, key):
    return ((a * S['xgb_probs'] + (1.0 - a) * S[key]) >= t).astype(int)

def tune_pfp(S, key):
    best = (-1, 1.0, 0.5)
    for a in alphas:
        for t in thresh_vals:
            f1 = f1_score(S['y_true'], soft_preds_pfp(S, a, t, key), pos_label=0, zero_division=0)
            if f1 > best[0]:
                best = (f1, a, t)
    return best

RISK_PARAMS = {}
for key, lbl in [('p_fp_5a', '5A desc-only'), ('p_fp_5b', '5B desc+features')]:
    f1, a, t = tune_pfp(TUNING, key)
    RISK_PARAMS[key] = (a, t)
    print(f'{lbl:18s} tuned on `tuning`: alpha={a:.2f} threshold={t:.2f} (tuning CO-F1={f1:.3f})')
    for name in SAMPLES:
        S = SAMPLES[name]
        p = soft_preds_pfp(S, a, t, key)
        S[f'{key}_preds'] = p
        m = metrics_dict(lbl, S['y_true'], p)
        print(f'    {name:11s} CO-F1={m["f1_co"]:.3f} recall={m["recall_co"]:.3f} prec={m["precision_co"]:.3f}')
    print()

# How novel is the text signal vs XGBoost? (correlation of risk with P(CO) on tuning)
r5a = np.corrcoef(TUNING['score_5a'], 1 - TUNING['xgb_probs'])[0, 1]
r5b = np.corrcoef(TUNING['score_5b'], 1 - TUNING['xgb_probs'])[0, 1]
print(f'[tuning] corr(5A risk, XGB P(CO))={r5a:+.3f} (low=novel text signal); '
      f'corr(5B, XGB P(CO))={r5b:+.3f} (high=LLM echoes the numbers)')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part 5 - Risk-score signal (tuning sample)', fontsize=12)
yt = TUNING['y_true']
ax1.hist(TUNING['score_5a'][yt == 0], bins=10, alpha=0.6, color='red', label='Actual CO')
ax1.hist(TUNING['score_5a'][yt == 1], bins=10, alpha=0.6, color='steelblue', label='Actual FP')
ax1.set_xlabel('5A risk score (0=safe,10=risky)'); ax1.set_ylabel('count'); ax1.set_title('5A score by true label'); ax1.legend(fontsize=8)
ax2.scatter(TUNING['score_5a'][yt == 1], TUNING['score_5b'][yt == 1], alpha=0.6, color='steelblue', label='Actual FP', s=30)
ax2.scatter(TUNING['score_5a'][yt == 0], TUNING['score_5b'][yt == 0], alpha=0.8, color='red', marker='^', label='Actual CO', s=40)
ax2.plot([0, 10], [0, 10], 'k--', alpha=0.3); ax2.set_xlabel('5A (desc only)'); ax2.set_ylabel('5B (desc+features)')
ax2.set_title('5A vs 5B'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '03_risk_scores.png'), dpi=150, bbox_inches='tight'); plt.show()


---
## Final leaderboard
Every strategy uses **parameters locked from `tuning`**. The cross-sample table shows CO-F1 on
all three; the **winning strategy is chosen by `robustness` (validation)**, and its **`test`**


In [ ]:
def all_strategy_preds(S):
    p = {
        'XGBoost (solo)': S['xgb_preds'],
        'LLM (solo)':     S['llm_binary'],
        'Soft blend (binary)':   S['soft_blend_preds'],
        'Confidence gate':       S['gate_preds'],
        '5A risk blend (desc)':  S['p_fp_5a_preds'],
        '5B risk blend (desc+feat)': S['p_fp_5b_preds'],
    }
    p.update(hard_label_preds(S))
    return p

# CO-F1 of every strategy on every sample.
f1_table = {}
for name in SAMPLES:
    S = SAMPLES[name]
    f1_table[name] = {strat: round(f1_score(S['y_true'], preds, pos_label=0, zero_division=0), 3)
                      for strat, preds in all_strategy_preds(S).items()}

lb = pd.DataFrame(f1_table)  # rows=strategy, cols=sample
lb = lb[['tuning', 'robustness']].sort_values('robustness', ascending=False)
print('Charged-Off F1 by strategy and sample (sorted by robustness/validation):\n')
print(lb.to_string())

winner = lb.index[0]
print(f'\n=== Winning strategy (by robustness): {winner} ===')
print(f'  tuning CO-F1     : {lb.loc[winner, "tuning"]:.3f}')
print(f'  robustness CO-F1 : {lb.loc[winner, "robustness"]:.3f}  (validation - used to pick)')
print(f'  TEST CO-F1       : {lb.loc[winner, "test"]:.3f}  <-- headline (touched once)')
print(f'\n  XGBoost solo on test: {lb.loc["XGBoost (solo)", "test"]:.3f}  '
      f'(delta {lb.loc[winner, "test"] - lb.loc["XGBoost (solo)", "test"]:+.3f})')
print(f'  LLM solo on test    : {lb.loc["LLM (solo)", "test"]:.3f}  '
      f'(delta {lb.loc[winner, "test"] - lb.loc["LLM (solo)", "test"]:+.3f})')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
order = lb.sort_values('test').index
y = np.arange(len(order)); h = 0.4
ax.barh(y + h/2, lb.loc[order, 'robustness'], h, color='#9c27b0', alpha=0.7, label='robustness (validation)')
ax.barh(y - h/2, lb.loc[order, 'test'],       h, color='#4caf50', alpha=0.85, label='test (headline)')
for yi, strat in enumerate(order):
    ax.text(lb.loc[strat, 'test']+0.003, yi - h/2, f"{lb.loc[strat,'test']:.3f}", va='center', fontsize=7)
ax.set_yticks(y); ax.set_yticklabels(order, fontsize=8)
ax.axvline(lb.loc['XGBoost (solo)', 'test'], color='#2196f3', ls='--', alpha=0.6, label='XGB solo (test)')
ax.set_xlabel('Charged-Off F1'); ax.set_title('Final leaderboard - held-out (locked params)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, '03_leaderboard.png'), dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# Persist results.
os.makedirs(RESULTS_DIR, exist_ok=True)
lb.to_csv(os.path.join(RESULTS_DIR, '03_blend_leaderboard.csv'))

locked = pd.DataFrame([
    {'strategy': 'soft blend (binary)', 'alpha': BLEND_A, 'threshold': BLEND_T, 'extra': ''},
    {'strategy': 'confidence gate',     'alpha': '',      'threshold': xgb_threshold, 'extra': f'band={GATE_BAND}'},
    {'strategy': '5A risk blend',       'alpha': RISK_PARAMS['p_fp_5a'][0], 'threshold': RISK_PARAMS['p_fp_5a'][1], 'extra': ''},
    {'strategy': '5B risk blend',       'alpha': RISK_PARAMS['p_fp_5b'][0], 'threshold': RISK_PARAMS['p_fp_5b'][1], 'extra': ''},
])
locked.to_csv(os.path.join(RESULTS_DIR, '03_locked_params.csv'), index=False)

print('Saved:')
print('  03_blend_leaderboard.csv   - CO-F1 by strategy x sample')
print('  03_locked_params.csv       - hyper-parameters tuned on `tuning`')
print('  06_groq_signals_*.csv      - cached per-loan Groq signals (one per sample)')
print('  03_baselines.png / 03_soft_blend_heatmap.png / 03_confidence_gate.png /')
print('  03_risk_scores.png / 03_leaderboard.png')
print(f'\nAll in: {RESULTS_DIR}')


---
## Part 7 — Sentence embeddings (optional appendix)
Encode borrower descriptions with `all-MiniLM-L6-v2`, reduce with PCA (unsupervised), and blend
the most label-correlated component with XGBoost. PC-blend weight is **tuned on `tuning`** and


In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.decomposition import PCA
    _ok = True
except ImportError:
    print('[SKIP] pip install sentence-transformers to run Part 7.')
    _ok = False

if _ok:
    enc = SentenceTransformer('all-MiniLM-L6-v2')

    def embed_pcs(df, pca=None):
        col = df['desc'] if 'desc' in df.columns else pd.Series([''] * len(df))
        descs = [str(d) if (pd.notna(d) and str(d).strip()) else '' for d in col]
        emb = enc.encode(descs, batch_size=32, show_progress_bar=False)
        if pca is None:
            pca = PCA(n_components=10, random_state=42).fit(emb)
        return pca.transform(emb), pca

    # Fit PCA + choose PC + tune weight on TUNING only.
    pcs_tune, pca = embed_pcs(TUNING['df'])
    rs = [np.corrcoef(pcs_tune[:, k], TUNING['y_true'])[0, 1] for k in range(pcs_tune.shape[1])]
    best_k = int(np.argmax(np.abs(rs))); sign = -1 if rs[best_k] < 0 else 1
    print(f'Most label-correlated PC on tuning: PC{best_k+1} (r={rs[best_k]:+.3f})')

    def pc_norm(pcs):
        v = sign * pcs[:, best_k]
        return (v - v.min()) / (v.max() - v.min() + 1e-9)

    pcn_tune = pc_norm(pcs_tune)
    best = (-1, 0.0, 0.5)
    for w in np.round(np.arange(0.0, 0.31, 0.02), 2):
        for t in thresh_vals:
            pr = (((1 - w) * TUNING['xgb_probs'] + w * pcn_tune) >= t).astype(int)
            f1 = f1_score(TUNING['y_true'], pr, pos_label=0, zero_division=0)
            if f1 > best[0]:
                best = (f1, w, t)
    _, PC_W, PC_T = best
    print(f'Tuned PC-blend: weight={PC_W:.2f} threshold={PC_T:.2f} (tuning CO-F1={best[0]:.3f})\n')

    for name in SAMPLES:
        S = SAMPLES[name]
        pcs, _ = embed_pcs(S['df'], pca=pca)
        pr = (((1 - PC_W) * S['xgb_probs'] + PC_W * pc_norm(pcs)) >= PC_T).astype(int)
        f1 = f1_score(S['y_true'], pr, pos_label=0, zero_division=0)
        print(f'  {name:11s} embedding-blend CO-F1={f1:.3f}  (XGB solo {f1_score(S["y_true"], S["xgb_preds"], pos_label=0, zero_division=0):.3f})')
    print('\nNote: PCA fit on tuning only; test transformed with the tuning-fit PCA - properly held out.')
